# Lab 05: Building a Backtest

A **backtest** is the process of testing a betting strategy against historical data to evaluate how it would have performed. This lab teaches you how to define a strategy, load historical odds from TimescaleDB, run a backtest, interpret key metrics (Sharpe, Sortino, max drawdown, profit factor), and visualize the equity curve.

By the end you will:

- Understand what a backtest is and why walk-forward validation matters
- Define a simple betting strategy with entry and exit rules
- Load historical odds data from TimescaleDB (or use synthetic data)
- Generate signals from your strategy
- Simulate trades at historical odds and track bankroll over time
- Compute performance metrics: Sharpe ratio, Sortino ratio, max drawdown, profit factor, win rate
- Visualize the equity curve with drawdown overlay
- Compare your strategy to a naive benchmark
- Run a walk-forward analysis to detect overfitting
- Compare two strategies head-to-head

---

## Prerequisites

- **Labs 01-04 completed** — you understand odds, EV, Kelly, and middling
- Familiarity with pandas DataFrames
- Understanding of basic statistics (mean, standard deviation)

### Key Concepts

| Concept | Description |
|---|---|
| **Backtest** | Replaying historical data through a strategy to measure performance |
| **Signal** | A trigger to place a bet (e.g., "home team EV > 0") |
| **Walk-forward** | Train on past data, test on future data (avoids look-ahead bias) |
| **Sharpe ratio** | Risk-adjusted return: (mean return) / (std of returns) |
| **Sortino ratio** | Like Sharpe but only penalizes downside volatility |
| **Max drawdown** | Largest peak-to-trough decline in your equity curve |
| **Profit factor** | Gross profits / gross losses (>1 is profitable)

---

## Section 1: Setup — Imports and DB Connection

We'll connect to TimescaleDB for historical odds and use the backtest and metrics modules from `sportsquant.core`.

In [ ]:
# Cell 4: Core imports
import asyncio
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sportsquant.core.betting.odds import Odds
from sportsquant.core.betting.engine import american_to_decimal, calculate_ev
from sportsquant.core.betting.kelly import KellyCalculator, BankrollManager
from sportsquant.core.betting.metrics import (
    calculate_sharpe_ratio,
    calculate_sortino_ratio,
    calculate_max_drawdown,
    calculate_profit_factor,
    calculate_streaks,
    calculate_performance_metrics,
    PerformanceMetrics,
)
from sportsquant.infra.db.connection import DBConfig, get_pool, reset_pool

import nest_asyncio
nest_asyncio.apply()

print('Imports loaded successfully.')

In [ ]:
# Cell 5: Connect to TimescaleDB (or use synthetic data)

db_available = False
try:
    config = DBConfig.from_env()
    pool = await get_pool(config)
    db_available = True
    print(f'Connected to {config.host}:{config.port}/{config.database}')
except Exception as e:
    print(f'DB not available: {e}')
    print('Will use synthetic data for this lab.')

---

## Section 2: What Is a Backtest?

A backtest answers the question: **"If I had applied this strategy to historical data, what would have happened?"**

The pipeline is:

```
Historical Odds → Strategy → Signals → Simulated Trades → P&L → Metrics
```

### Walk-Forward vs. Train/Test Split

| Approach | Description | Pros | Cons |
|---|---|---|---|
| **Train/Test Split** | Fit parameters on 70% of data, test on 30% | Simple | Doesn't capture temporal effects |
| **Walk-Forward** | Use expanding window: train on weeks 1-N, test on week N+1 | Realistic | Computationally expensive |
| **K-Fold CV** | Random splits across time periods | Fast | Can leak future data |

For betting, **walk-forward** is the gold standard because odds and markets evolve over time.

---

## Section 3: The Backtest Architecture

SportsQuant's backtest pipeline has these components:

| Component | Module | Purpose |
|---|---|---|
| Strategy | `core.betting.strategies` | Define when to bet |
| Signal | Output of strategy | A tuple of (side, odds, EV) |
| Trade simulation | Inline in this lab | Apply signals at historical odds |
| P&L tracking | Inline | Track cumulative profit/loss |
| Metrics | `core.betting.metrics` | Sharpe, Sortino, drawdown, etc. |
| Visualization | `models.predictive.viz.backtest_viz` | Equity curve, drawdown chart |

---

## Section 4: Load Historical Odds

We need historical odds data to replay through our strategy. We'll query `odds_ticks` from TimescaleDB. If the DB is unavailable, we'll generate realistic synthetic data.

In [ ]:
# Cell 9: Load historical odds data
#
# We need a DataFrame with columns:
#   event_id, timestamp, home_team, away_team, market, selection,
#   price (American odds), line (spread or total value), implied_prob

if db_available:
    odds_rows = await pool.fetch(
        "SELECT sport, league, event_id, book, market, selection, price, line, ts "
        "FROM odds_ticks WHERE market IN ('h2h', 'spreads') "
        "ORDER BY ts DESC LIMIT 500"
    )
    if odds_rows:
        odds_data = [dict(r) for r in odds_rows]
        print(f'Loaded {len(odds_data)} rows from DB')
    else:
        print('No odds data in DB. Using synthetic data.')
        db_available = False

if not db_available:
    # Generate realistic synthetic odds data for 200 NFL games
    np.random.seed(42)
    N_GAMES = 200
    teams = ['KC', 'BUF', 'SF', 'DAL', 'BAL', 'DET', 'PHI', 'MIA', 'CIN', 'LAR']
    books = ['draftkings', 'fanduel', 'betmgm']
    
    odds_data = []
    for i in range(N_GAMES):
        home = teams[i % len(teams)]
        away = teams[(i + 3) % len(teams)]
        if home == away:
            away = teams[(i + 5) % len(teams)]
        event_id = f'{home}_vs_{away}_2024_w{(i % 18) + 1}'
        week = (i % 18) + 1
        
        # Spread: home team spread centered around -3 (home field advantage)
        base_spread = np.random.normal(-3, 4)
        for book in books:
            spread_line = round(base_spread + np.random.normal(0, 0.5), 1)
            odds_data.append({
                'sport': 'nfl',
                'league': 'NFL',
                'event_id': event_id,
                'book': book,
                'market': 'spreads',
                'selection': home,
                'price': -110 + np.random.randint(-10, 11),
                'line': spread_line,
                'ts': f'2024-09-{min(week + 4, 28):02d}T18:00:00Z',
                'home_team': home,
                'away_team': away,
                'week': week,
            })
        
        # H2H (moneyline)
        implied_home = 0.4 + np.random.random() * 0.3  # 40-70% home win prob
        for book in books:
            odds_data.append({
                'sport': 'nfl',
                'league': 'NFL',
                'event_id': event_id,
                'book': book,
                'market': 'h2h',
                'selection': home,
                'price': -110 + np.random.randint(-30, 31),
                'line': None,
                'ts': f'2024-09-{min(week + 4, 28):02d}T18:00:00Z',
                'home_team': home,
                'away_team': away,
                'week': week,
                'implied_home_prob': implied_home,
            })
    
    # Generate game outcomes (for backtest simulation)
    outcomes = []
    for i in range(N_GAMES):
        home = teams[i % len(teams)]
        away = teams[(i + 3) % len(teams)]
        if home == away:
            away = teams[(i + 5) % len(teams)]
        home_win = np.random.random() < 0.57  # home teams win ~57% in NFL
        margin = abs(np.random.normal(6, 13.5))  # NFL margin distribution
        home_score = int(np.random.normal(22, 6))
        away_score = int(np.random.normal(20, 6))
        if not home_win:
            home_score, away_score = away_score, home_score
        outcomes.append({
            'event_id': f'{home}_vs_{away}_2024_w{(i % 18) + 1}',
            'home_win': home_win,
            'home_score': max(home_score, 7),
            'away_score': max(away_score, 7),
            'margin': home_score - away_score,
        })
    
    print(f'Generated {len(odds_data)} synthetic odds rows for {N_GAMES} games')
    print(f'Generated {len(outcomes)} game outcomes')

odds_df = pd.DataFrame(odds_data)
outcomes_df = pd.DataFrame(outcomes)
print(f'\nOdds DataFrame shape: {odds_df.shape}')
print(f'Outcomes DataFrame shape: {outcomes_df.shape}')
print(f'\nOdds columns: {list(odds_df.columns)}')
print(f'\nFirst 5 odds rows:')
print(odds_df.head())

---

## Section 5: Define a Simple Strategy

Our strategy: **Bet the home team on the moneyline if the implied probability is below 55% and the home team spread is less than +3.**

The idea is that home underdogs (small spread) are often undervalued. This is a simplified version of strategies used by quantitative bettors.

In [ ]:
# Cell 11: Define the home underdog strategy
#
# Strategy rules:
# 1. Look at h2h (moneyline) odds for the home team
# 2. Check if implied probability < 0.55 (the home team is an underdog or slight favorite)
# 3. Check if the spread is between +1 and -3 (close game, home field advantage)
# 4. If both conditions met, bet the home team

from sportsquant.util.time_utils import american_to_implied_prob

@dataclass
class BetSignal:
    """A signal to place a bet."""
    event_id: str
    selection: str
    market: str
    price: int  # American odds
    implied_prob: float
    ev: float
    side: str  # 'home' or 'away'


def generate_signals(
    odds_df: pd.DataFrame,
    outcomes_df: pd.DataFrame,
    max_implied_prob: float = 0.55,
    max_spread: float = 3.0,
) -> list[BetSignal]:
    """Generate bet signals from odds data using the home underdog strategy.

    Args:
        odds_df: DataFrame with odds data.
        outcomes_df: DataFrame with game outcomes.
        max_implied_prob: Maximum implied probability to consider (underdog threshold).
        max_spread: Maximum absolute spread to consider (close game threshold).

    Returns:
        List of BetSignal objects.
    """
    signals = []
    
    # Get unique events
    if 'event_id' not in odds_df.columns:
        return signals
    events = odds_df['event_id'].unique()
    
    for event_id in events:
        event_odds = odds_df[odds_df['event_id'] == event_id]
        
        # Get h2h (moneyline) odds for home team
        h2h = event_odds[event_odds['market'] == 'h2h']
        if h2h.empty:
            continue
        
        # Use the first book's odds
        home_row = h2h.iloc[0]
        price = int(home_row['price'])
        
        # Calculate implied probability
        try:
            implied_prob = american_to_implied_prob(price)
        except (ValueError, ZeroDivisionError):
            continue
        
        # Get the spread for this game
        spreads = event_odds[event_odds['market'] == 'spreads']
        if not spreads.empty:
            spread_line = float(spreads.iloc[0]['line'])
        else:
            # No spread data, use 0 as neutral
            spread_line = 0.0
        
        # Strategy: bet home team if implied_prob < threshold and spread is close
        if implied_prob < max_implied_prob and abs(spread_line) <= max_spread:
            # Calculate EV
            decimal_odds = american_to_decimal(price)
            # Use true probability from outcomes if available
            event_outcomes = outcomes_df[outcomes_df['event_id'] == event_id]
            if not event_outcomes.empty:
                true_prob = float(event_outcomes.iloc[0]['home_win'])
            else:
                true_prob = 0.55  # Default NFL home win rate
            
            ev = calculate_ev(true_prob, decimal_odds)
            
            signals.append(BetSignal(
                event_id=event_id,
                selection=str(home_row.get('selection', home_row.get('home_team', 'home'))),
                market='h2h',
                price=price,
                implied_prob=implied_prob,
                ev=ev,
                side='home',
            ))
    
    return signals


signals = generate_signals(odds_df, outcomes_df)
print(f'Generated {len(signals)} bet signals')
for s in signals[:5]:
    print(f'  {s.event_id}: {s.selection} @ {s.price}, implied_prob={s.implied_prob:.3f}, ev={s.ev:.4f}')

---

## Section 6: Simulate Trades

For each signal, we simulate placing a bet at the historical odds. We use flat betting (1 unit per bet) for simplicity, then compare to Kelly sizing.

In [ ]:
# Cell 13: Simulate trades from signals
#
# For each signal, check the actual outcome and compute P&L.
# We use flat betting: 1 unit per bet.

def simulate_trades(
    signals: list[BetSignal],
    outcomes_df: pd.DataFrame,
    unit_size: float = 1.0,
) -> pd.DataFrame:
    """Simulate trades from bet signals against actual outcomes.

    Args:
        signals: List of BetSignal objects.
        outcomes_df: DataFrame with game outcomes.
        unit_size: Size of each bet in units.

    Returns:
        DataFrame with bet results.
    """
    trades = []
    
    for signal in signals:
        # Find the outcome for this event
        event_outcome = outcomes_df[outcomes_df['event_id'] == signal.event_id]
        if event_outcome.empty:
            continue
        
        home_win = bool(event_outcome.iloc[0]['home_win'])
        
        # Determine if our bet won
        won = (signal.side == 'home' and home_win)
        
        # Calculate P&L
        decimal_odds = american_to_decimal(signal.price)
        if won:
            pnl = unit_size * (decimal_odds - 1)  # Profit
        else:
            pnl = -unit_size  # Loss
        
        trades.append({
            'event_id': signal.event_id,
            'selection': signal.selection,
            'market': signal.market,
            'price': signal.price,
            'side': signal.side,
            'decimal_odds': decimal_odds,
            'implied_prob': signal.implied_prob,
            'ev': signal.ev,
            'won': won,
            'pnl': pnl,
            'stake': unit_size,
        })
    
    return pd.DataFrame(trades)


trades_df = simulate_trades(signals, outcomes_df, unit_size=1.0)
print(f'Simulated {len(trades_df)} trades')
if not trades_df.empty:
    wins = trades_df['won'].sum()
    losses = len(trades_df) - wins
    print(f'  Wins: {wins}, Losses: {losses}')
    print(f'  Win rate: {wins / len(trades_df):.2%}')
    print(f'  Total P&L: ${trades_df["pnl"].sum():.2f}')
    print(f'  Average P&L per bet: ${trades_df["pnl"].mean():.4f}')
else:
    print('No trades generated. Adjusting strategy parameters...')
    # Try with wider parameters
    signals = generate_signals(odds_df, outcomes_df, max_implied_prob=0.65, max_spread=7.0)
    trades_df = simulate_trades(signals, outcomes_df, unit_size=1.0)
    print(f'  Adjusted: {len(trades_df)} trades, win rate: {trades_df["won"].mean():.2%}')

---

## Section 7: Track Bankroll Over Time

The equity curve shows how your bankroll changes over the course of the backtest. Starting with a $1,000 bankroll, we add/subtract each bet's P&L.

In [ ]:
# Cell 15: Compute bankroll over time (equity curve)

INITIAL_BANKROLL = 1_000.0

if not trades_df.empty:
    trades_df = trades_df.copy()
    trades_df['cumulative_pnl'] = trades_df['pnl'].cumsum()
    trades_df['bankroll'] = INITIAL_BANKROLL + trades_df['cumulative_pnl']
    trades_df['bet_number'] = range(1, len(trades_df) + 1)
    
    print(f'Equity curve summary:')
    print(f'  Starting bankroll: ${INITIAL_BANKROLL:,.2f}')
    print(f'  Final bankroll:    ${trades_df["bankroll"].iloc[-1]:,.2f}')
    print(f'  Peak bankroll:     ${trades_df["bankroll"].max():,.2f}')
    print(f'  Min bankroll:      ${trades_df["bankroll"].min():,.2f}')
    print(f'  Total return:      {(trades_df["bankroll"].iloc[-1] / INITIAL_BANKROLL - 1) * 100:.2f}%')
else:
    print('No trades to compute equity curve.')

---

## Section 8: Compute Performance Metrics

We use SportsQuant's built-in metrics module to compute Sharpe ratio, Sortino ratio, max drawdown, profit factor, win rate, and streak statistics.

In [ ]:
# Cell 17: Compute comprehensive performance metrics

if not trades_df.empty:
    # Use the calculate_performance_metrics function from sportsquant
    metrics = calculate_performance_metrics(
        pnl_series=trades_df['pnl'],
        ev_series=trades_df['ev'],
        outcome_series=trades_df['won'],
        stake_series=trades_df['stake'],
        risk_free_rate=0.0,
    )
    
    print('Performance Metrics:')
    print(f'  Total Bets:       {metrics.core.total_bets}')
    print(f'  Win Rate:         {metrics.core.win_rate:.2%}')
    print(f'  Total P&L:        ${metrics.core.total_pnl:.2f}')
    print(f'  Avg P&L per Bet:  ${metrics.core.mean_pnl_per_bet:.4f}')
    print(f'  Avg EV per Bet:   ${metrics.core.mean_ev_per_bet:.4f}')
    print()
    print('Risk-Adjusted Metrics:')
    print(f'  Sharpe Ratio:     {metrics.risk_adjusted.sharpe_ratio:.4f}')
    print(f'  Sortino Ratio:     {metrics.risk_adjusted.sortino_ratio:.4f}')
    print(f'  Calmar Ratio:      {metrics.risk_adjusted.calmar_ratio:.4f}')
    print(f'  Profit Factor:     {metrics.risk_adjusted.profit_factor:.4f}')
    print()
    print('Drawdown Metrics:')
    print(f'  Max Drawdown:      ${metrics.drawdown.max_drawdown:.2f}')
    print(f'  Max Drawdown %:    {metrics.drawdown.max_drawdown_pct:.2f}%')
    print(f'  DD Duration:       {metrics.drawdown.max_drawdown_duration} bets')
    print()
    print('Streak Metrics:')
    print(f'  Max Win Streak:    {metrics.streaks.max_win_streak}')
    print(f'  Max Loss Streak:   {metrics.streaks.max_loss_streak}')
    print(f'  Current Streak:    {metrics.streaks.current_streak} ({metrics.streaks.current_streak_type})')
    print()
    print('Distribution Metrics:')
    print(f'  P&L Std Dev:      ${metrics.distribution.pnl_std:.4f}')
    print(f'  P&L Skewness:      {metrics.distribution.pnl_skewness:.4f}')
    print(f'  P&L Kurtosis:      {metrics.distribution.pnl_kurtosis:.4f}')
    print()
    print('Return Metrics:')
    print(f'  Total Return %:    {metrics.returns.total_return_pct:.2f}%')
    print(f'  Return on Risk:     {metrics.returns.return_on_risk:.4f}')
else:
    print('No trades to compute metrics.')

---

## Section 9: Visualize the Equity Curve

A picture is worth a thousand numbers. We plot the equity curve (bankroll over time) and the drawdown chart below it.

In [ ]:
# Cell 19: Plot equity curve with drawdown

if not trades_df.empty:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                    gridspec_kw={'height_ratios': [2, 1]})
    
    # Top: Equity curve
    bet_numbers = trades_df['bet_number'].values
    bankroll = trades_df['bankroll'].values
    
    ax1.plot(bet_numbers, bankroll, linewidth=2, color='#2E86AB')
    ax1.fill_between(bet_numbers, bankroll, INITIAL_BANKROLL,
                     where=(bankroll >= INITIAL_BANKROLL), alpha=0.3, color='green')
    ax1.fill_between(bet_numbers, bankroll, INITIAL_BANKROLL,
                     where=(bankroll < INITIAL_BANKROLL), alpha=0.3, color='red')
    ax1.axhline(y=INITIAL_BANKROLL, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax1.set_ylabel('Bankroll ($)', fontsize=12)
    ax1.set_title('Backtest Equity Curve — Home Underdog Strategy', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Bottom: Drawdown chart
    cum_pnl = trades_df['cumulative_pnl'].values
    running_max = np.maximum.accumulate(cum_pnl)
    drawdown = running_max - cum_pnl
    drawdown_pct = np.where(running_max > 0, (drawdown / running_max) * 100, 0)
    
    ax2.fill_between(bet_numbers, -drawdown_pct, 0, alpha=0.5, color='#E63946')
    ax2.plot(bet_numbers, -drawdown_pct, linewidth=1.5, color='darkred')
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax2.set_xlabel('Bet Number', fontsize=12)
    ax2.set_ylabel('Drawdown (%)', fontsize=12)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f'\nPeak bankroll: ${trades_df["bankroll"].max():,.2f}')
    print(f'Max drawdown: ${metrics.drawdown.max_drawdown:.2f} ({metrics.drawdown.max_drawdown_pct:.2f}%)')
else:
    print('No trades to visualize.')

---

## Section 10: Compare to a Benchmark Strategy

Every strategy needs a benchmark. We'll compare our home underdog strategy against a naive "bet the favorite every time" strategy.

In [ ]:
# Cell 21: Benchmark strategy — bet the favorite every time
#
# A naive benchmark: always bet on the team with the lower (more negative) spread,
# i.e., the favorite. This has zero intelligence but serves as a baseline.

def generate_benchmark_signals(
    odds_df: pd.DataFrame,
    outcomes_df: pd.DataFrame,
) -> list[BetSignal]:
    """Bet the favorite (lower spread) every time."""
    signals = []
    
    if 'event_id' not in odds_df.columns:
        return signals
    
    events = odds_df['event_id'].unique()
    
    for event_id in events:
        event_odds = odds_df[odds_df['event_id'] == event_id]
        
        # Get h2h odds
        h2h = event_odds[event_odds['market'] == 'h2h']
        if h2h.empty:
            continue
        
        home_row = h2h.iloc[0]
        price = int(home_row['price'])
        
        try:
            implied_prob = american_to_implied_prob(price)
        except (ValueError, ZeroDivisionError):
            continue
        
        # Get spreads
        spreads = event_odds[event_odds['market'] == 'spreads']
        if not spreads.empty:
            spread_line = float(spreads.iloc[0]['line'])
        else:
            spread_line = 0.0
        
        # Bet the favorite (team with negative spread)
        side = 'home' if spread_line < 0 else 'away'
        
        # If spread is pick'em, skip
        if abs(spread_line) < 0.5:
            continue
        
        decimal_odds = american_to_decimal(price)
        ev = calculate_ev(implied_prob, decimal_odds)
        
        signals.append(BetSignal(
            event_id=event_id,
            selection=str(home_row.get('selection', side)),
            market='h2h',
            price=price,
            implied_prob=implied_prob,
            ev=ev,
            side=side,
        ))
    
    return signals


benchmark_signals = generate_benchmark_signals(odds_df, outcomes_df)
benchmark_trades = simulate_trades(benchmark_signals, outcomes_df, unit_size=1.0)

if not benchmark_trades.empty:
    benchmark_trades['cumulative_pnl'] = benchmark_trades['pnl'].cumsum()
    benchmark_trades['bankroll'] = INITIAL_BANKROLL + benchmark_trades['cumulative_pnl']
    benchmark_trades['bet_number'] = range(1, len(benchmark_trades) + 1)
    
    benchmark_metrics = calculate_performance_metrics(
        pnl_series=benchmark_trades['pnl'],
        ev_series=benchmark_trades['ev'],
        outcome_series=benchmark_trades['won'],
        stake_series=benchmark_trades['stake'],
    )
    
    print('Benchmark Strategy (Bet the Favorite):')
    print(f'  Total Bets:  {benchmark_metrics.core.total_bets}')
    print(f'  Win Rate:    {benchmark_metrics.core.win_rate:.2%}')
    print(f'  Total P&L:   ${benchmark_metrics.core.total_pnl:.2f}')
    print(f'  Sharpe:      {benchmark_metrics.risk_adjusted.sharpe_ratio:.4f}')
    print(f'  Profit Factor: {benchmark_metrics.risk_adjusted.profit_factor:.4f}')
else:
    print('No benchmark trades generated.')

In [ ]:
# Cell 22: Compare strategies head-to-head

if not trades_df.empty and not benchmark_trades.empty:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    ax.plot(trades_df['bet_number'], trades_df['bankroll'],
            linewidth=2, color='#2E86AB', label='Home Underdog Strategy')
    ax.plot(benchmark_trades['bet_number'], benchmark_trades['bankroll'],
            linewidth=2, color='#E63946', alpha=0.7, label='Bet the Favorite (Benchmark)')
    ax.axhline(y=INITIAL_BANKROLL, color='black', linestyle='--', linewidth=1, alpha=0.5)
    
    ax.set_xlabel('Bet Number', fontsize=12)
    ax.set_ylabel('Bankroll ($)', fontsize=12)
    ax.set_title('Strategy Comparison: Home Underdog vs. Bet the Favorite', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary comparison
    print('\nStrategy Comparison:')
    print(f"{'Metric':<20} {'Home Underdog':>15} {'Benchmark':>15}")
    print(f"{'-'*20} {'-'*15} {'-'*15}")
    print(f"{'Total Bets':<20} {metrics.core.total_bets:>15} {benchmark_metrics.core.total_bets:>15}")
    print(f"{'Win Rate':<20} {metrics.core.win_rate:>14.2%} {benchmark_metrics.core.win_rate:>14.2%}")
    print(f"{'Total P&L':<20} {metrics.core.total_pnl:>15.2f} {benchmark_metrics.core.total_pnl:>15.2f}")
    print(f"{'Sharpe Ratio':<20} {metrics.risk_adjusted.sharpe_ratio:>15.4f} {benchmark_metrics.risk_adjusted.sharpe_ratio:>15.4f}")
    print(f"{'Profit Factor':<20} {metrics.risk_adjusted.profit_factor:>15.4f} {benchmark_metrics.risk_adjusted.profit_factor:>15.4f}")
    print(f"{'Max Drawdown $':<20} {metrics.drawdown.max_drawdown:>15.2f} {benchmark_metrics.drawdown.max_drawdown:>15.2f}")
else:
    print('Insufficient data for comparison.')

---

## Section 11: Walk-Forward Analysis

A walk-forward analysis trains on the first N bets and tests on bet N+1, then expands the training window. This avoids look-ahead bias and reveals whether a strategy is overfit.

We'll split our backtest data into windows and compute rolling metrics.

In [ ]:
# Cell 24: Walk-forward analysis
#
# Split trades into windows and compute metrics for each.
# This simulates the real-world scenario where you don't know future results.

if not trades_df.empty and len(trades_df) >= 20:
    WINDOW_SIZE = max(10, len(trades_df) // 5)  # 5 windows
    n_windows = len(trades_df) // WINDOW_SIZE
    
    print(f'Walk-Forward Analysis ({n_windows} windows of {WINDOW_SIZE} bets each):')
    print()
    print(f"{'Window':<8} {'Bets':<8} {'Win Rate':<12} {'P&L':<10} {'Sharpe':<10} {'Max DD':<10}")
    print(f"{'-'*8} {'-'*8} {'-'*12} {'-'*10} {'-'*10} {'-'*10}")
    
    window_metrics = []
    for i in range(n_windows):
        start = i * WINDOW_SIZE
        end = start + WINDOW_SIZE
        window = trades_df.iloc[start:end]
        
        if len(window) < 5:
            continue
        
        w_metrics = calculate_performance_metrics(
            pnl_series=window['pnl'],
            ev_series=window['ev'],
            outcome_series=window['won'],
            stake_series=window['stake'],
        )
        
        window_metrics.append(w_metrics)
        print(f'{i+1:<8} {len(window):<8} {w_metrics.core.win_rate:<12.2%} '
              f'{w_metrics.core.total_pnl:<10.2f} {w_metrics.risk_adjusted.sharpe_ratio:<10.4f} '
              f'{w_metrics.drawdown.max_drawdown:<10.2f}')
    
    # Check for consistency
    if window_metrics:
        win_rates = [m.core.win_rate for m in window_metrics]
        sharpes = [m.risk_adjusted.sharpe_ratio for m in window_metrics]
        print(f'\nConsistency Check:')
        print(f'  Win rate range: {min(win_rates):.2%} - {max(win_rates):.2%}')
        print(f'  Sharpe range: {min(sharpes):.4f} - {max(sharpes):.4f}')
        print(f'  Strategy is {"CONSISTENT" if max(win_rates) - min(win_rates) < 0.3 else "INCONSISTENT"} across windows')
else:
    print('Insufficient trades for walk-forward analysis. Need at least 20 trades.')

---

## Section 12: A Second Strategy — Middling

Now let's implement the middling strategy from Lab 04 as a backtest. We'll simulate betting on detected middles and compare the results to our home underdog strategy.

In [ ]:
# Cell 26: Middling strategy backtest
#
# We simulate a simple middling strategy:
# For each game, if we detect a middle of >= 1 point across books,
# bet both sides at -110.

from sportsquant.core.betting.strategies.middling import detect_spread_middles

# Prepare data for middling detection
if 'market' in odds_df.columns and not odds_df[odds_df['market'] == 'spreads'].empty:
    spread_for_detect = odds_df[odds_df['market'] == 'spreads'].rename(columns={
        'event_id': 'game_id',
        'line': 'spread_home',
        'book': 'source_id',
    })[['game_id', 'spread_home', 'source_id']]
    
    middles = detect_spread_middles(spread_for_detect, min_middle_points=1.0)
    print(f'Detected {len(middles)} spread middles >= 1 point')
else:
    middles = pd.DataFrame()
    print('No spread data available for middling.')

# Simulate middling trades
middle_trades = []
if not middles.empty:
    for _, row in middles.iterrows():
        # For each middle, simulate betting both sides at -110
        # Outcome: if margin is in the middle range, both win; otherwise, one wins
        event_id = row['game_id']
        event_outcome = outcomes_df[outcomes_df['event_id'] == event_id]
        
        if event_outcome.empty:
            continue
        
        margin = float(event_outcome.iloc[0]['margin'])
        low_line = row['low_line']
        high_line = row['high_line']
        
        # Check if margin falls in the middle
        low_abs = abs(low_line)
        high_abs = abs(high_line)
        
        # Bet both sides at -110 (decimal odds ~1.91)
        dec_odds = 1.909  # -110 in decimal
        
        if low_abs < margin < high_abs:
            # Both bets win!
            pnl = 2 * (dec_odds - 1)  # Profit on both legs
            won = True
        elif margin == low_abs or margin == high_abs:
            # Push on one side, win on other
            pnl = (dec_odds - 1) - 1  # Net: win one, lose one
            won = False
        else:
            # One wins, one loses
            pnl = -0.1 * 2  # Loss of vig on both bets
            won = False
        
        middle_trades.append({
            'event_id': event_id,
            'low_line': low_line,
            'high_line': high_line,
            'middle_pts': row['middle_points'],
            'margin': margin,
            'pnl': pnl,
            'won': won,
            'stake': 2.0,  # 1 unit on each leg
            'ev': 0.0,  # Will compute below
        })

middle_trades_df = pd.DataFrame(middle_trades)
if not middle_trades_df.empty:
    middle_trades_df['cumulative_pnl'] = middle_trades_df['pnl'].cumsum()
    middle_trades_df['bankroll'] = INITIAL_BANKROLL + middle_trades_df['cumulative_pnl']
    
    mid_metrics = calculate_performance_metrics(
        pnl_series=middle_trades_df['pnl'],
        ev_series=middle_trades_df['ev'],
        outcome_series=middle_trades_df['won'],
        stake_series=middle_trades_df['stake'],
    )
    
    print(f'\nMiddling Strategy Results:')
    print(f'  Total Middles: {len(middle_trades_df)}')
    print(f'  Win Rate:       {mid_metrics.core.win_rate:.2%}')
    print(f'  Total P&L:      ${mid_metrics.core.total_pnl:.2f}')
    print(f'  Sharpe:         {mid_metrics.risk_adjusted.sharpe_ratio:.4f}')
    print(f'  Profit Factor:  {mid_metrics.risk_adjusted.profit_factor:.4f}')
else:
    print('No middle trades generated.')
    print('This is expected if no middles were detected in the data.')

In [ ]:
# Cell 27: Three-strategy comparison

if not trades_df.empty:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    ax.plot(trades_df['bet_number'], trades_df['bankroll'],
            linewidth=2, color='#2E86AB', label='Home Underdog')
    
    if not benchmark_trades.empty:
        ax.plot(benchmark_trades['bet_number'], benchmark_trades['bankroll'],
                linewidth=2, color='#E63946', alpha=0.7, label='Bet Favorite')
    
    if not middle_trades_df.empty:
        ax.plot(range(1, len(middle_trades_df) + 1), middle_trades_df['bankroll'],
                linewidth=2, color='#2A9D8F', label='Middling')
    
    ax.axhline(y=INITIAL_BANKROLL, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('Bet Number', fontsize=12)
    ax.set_ylabel('Bankroll ($)', fontsize=12)
    ax.set_title('Three-Strategy Comparison', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print('Insufficient data for three-strategy comparison.')

---

## Section 13: Using SportsQuant's Backtest Module

SportsQuant has a built-in backtest engine (`sportsquant.core.backtest`) that handles the full pipeline: loading lines, making decisions, tracking P&L, and computing metrics. Let's see how it works.

In [ ]:
# Cell 29: Demonstrate SportsQuant's built-in backtest module
#
# The backtest module provides:
# - PraBacktestConfig: configuration for PRA prop backtests
# - backtest_pra_lines(): full backtest pipeline
# - WalkForwardBacktest: walk-forward validation
# - RegimeAwareBacktest: regime-aware backtesting

from sportsquant.core.backtest import (
    PraBacktestConfig,
    backtest_pra_lines,
    WalkForwardConfig,
    WalkForwardBacktest,
)

print('SportsQuant Backtest Module:')
print(f'  PraBacktestConfig: {PraBacktestConfig.__name__}')
print(f'  backtest_pra_lines: {backtest_pra_lines.__name__}')
print(f'  WalkForwardConfig: {WalkForwardConfig.__name__}')
print(f'  WalkForwardBacktest: {WalkForwardBacktest.__name__}')
print()
print('These modules are designed for PRA (Points + Rebounds + Assists) prop backtests.')
print('For custom strategies like our home underdog, the manual approach above gives')
print('more flexibility, while the built-in module handles CSV-based backtests automatically.')
print()
print('Key config options for PraBacktestConfig:')
config_example = PraBacktestConfig(
    league='NFL',
    season='2024',
    cache_root=Path('./cache'),
    lines_csv=Path('./data/nfl_lines.csv'),
)
for field_name, field_value in config_example.__dict__.items():
    print(f'  {field_name}: {field_value}')

---

## Section 14: Walk-Forward with SportsQuant

The `WalkForwardBacktest` class provides a production-ready walk-forward backtesting framework. It uses an expanding window: train on weeks 1-N, test on week N+1.

In [ ]:
# Cell 31: Walk-forward configuration
#
# WalkForwardConfig specifies:
# - train_window: Number of periods to train on
# - test_window: Number of periods to test on
# - min_train_samples: Minimum samples required to train
# - step_size: How many periods to advance each fold

wf_config = WalkForwardConfig(
    train_window=82,
    test_window=20,
    min_train_samples=50,
    step_size=1,
)

print('Walk-Forward Configuration:')
print(f'  Train window: {wf_config.train_window} bets')
print(f'  Test window:  {wf_config.test_window} bets')
print(f'  Min samples:  {wf_config.min_train_samples}')
print(f'  Step size:    {wf_config.step_size}')
print()
print('The WalkForwardBacktest class handles:')
print('  - Expanding window training')
print('  - Per-fold model retraining')
print('  - Metric aggregation across folds')
print('  - Regime-aware splitting (optional)')
print()
print('For custom strategies, you can manually implement walk-forward as shown in Section 11.')

---

## Exercises

Try these on your own:

1. **Tune the strategy parameters** — Modify `max_implied_prob` and `max_spread` in `generate_signals()`. What happens if you increase the implied probability threshold to 0.65? What about decreasing the spread threshold to 1.5?

2. **Add transaction costs** — Real sportsbooks charge vig (typically 4.5-5% on each bet). Modify the simulation to deduct a 5% transaction cost from each bet. How does this affect the Sharpe ratio?

3. **Test on a different sport** — Modify the synthetic data generator to create NBA or MLB odds. Re-run the backtest. How do the metrics differ across sports?

4. **Kelly sizing** — Instead of flat betting, use the Kelly Criterion to size each bet. Compare the equity curve to the flat-betting approach. Is Kelly more volatile?

5. **Use real data** — If you have a TimescaleDB instance with historical odds, modify the odds loading section to pull real data instead of synthetic. How do real results compare?

---

## Summary

In this lab you learned:

- What a backtest is and why walk-forward validation is important
- How to define a strategy and generate bet signals
- How to simulate trades at historical odds and track P&L
- How to compute performance metrics: Sharpe ratio, Sortino ratio, max drawdown, profit factor
- How to visualize the equity curve with drawdown overlay
- How to compare strategies against a benchmark
- How to run a walk-forward analysis to detect overfitting
- How to use SportsQuant's built-in backtest and metrics modules

### Key API Reference

| Function/Class | Module | Purpose |
|---|---|---|
| `calculate_sharpe_ratio()` | `sportsquant.core.betting.metrics` | Sharpe ratio |
| `calculate_sortino_ratio()` | `sportsquant.core.betting.metrics` | Sortino ratio |
| `calculate_max_drawdown()` | `sportsquant.core.betting.metrics` | Max drawdown |
| `calculate_profit_factor()` | `sportsquant.core.betting.metrics` | Profit factor |
| `calculate_performance_metrics()` | `sportsquant.core.betting.metrics` | All metrics at once |
| `PraBacktestConfig` | `sportsquant.core.backtest` | PRA backtest config |
| `WalkForwardConfig` | `sportsquant.core.backtest` | Walk-forward config |
| `WalkForwardBacktest` | `sportsquant.core.backtest` | Walk-forward engine |

### Key Concepts

| Concept | Description |
|---|---|
| **Backtest** | Replaying historical data through a strategy |
| **Signal** | A trigger to place a bet |
| **Equity curve** | Bankroll over time |
| **Sharpe ratio** | Risk-adjusted return (higher is better) |
| **Sortino ratio** | Downside risk-adjusted return |
| **Max drawdown** | Worst peak-to-trough decline |
| **Profit factor** | Gross profit / gross loss |
| **Walk-forward** | Train-on-past, test-on-future validation |

### Next Steps

Continue to **Lab 06: NFL Game Prediction (XGBoost)** to learn how to build a predictive model for NFL game outcomes.

---

*Don't forget to close the pool:*
```python
await pool.close()
```

In [ ]:
# Cell 34: Close the connection pool
if db_available:
    await pool.close()
    print('Connection pool closed. Lab 05 complete!')
else:
    print('Lab 05 complete! (used synthetic data, no DB connection to close)')